In [ ]:
import pandas as pd
import numpy as np
import itertools
import glob
import sys
import os

from models_generation_utils import *

In [ ]:
filepath = '../../data/Observatory_paper_mendeley.xlsx'

# SEV

In [ ]:
wwtp = 'SEV'
raw_data = get_raw_data(wwtp, filepath)
obs_matrix = get_obs_matrix(wwtp, gather_CIs=False)
print(raw_data.shape, obs_matrix.shape)

In [ ]:
# 'NH4', 'DCO', 'DBO', 'NTK', 'NGL', 'PT', 'MES'
QIj_vect = np.array([10.3, 120, 60, 12.6, 12.6, 1.2, 60])

In [ ]:
elements = np.arange(obs_matrix.shape[1]).tolist()

combis = []
for r in range(1, len(elements)+1):
    combis.extend(itertools.combinations(elements, r))

### Model 1

In [ ]:
model_1x = model_1_estimation(obs_matrix[:366], QIj_vect, combis)

### Model 2

In [ ]:
raw_obs_matrix = raw_data.loc[::, ['NH4', 'DCO', 'DBO', 'NTK', 'NGL', 'PT', 'MES', 'plantVolume']]
for k in range(7):
    raw_obs_matrix.iloc[:, k] *= raw_obs_matrix.iloc[:, -1]

raw_obs_matrix = raw_obs_matrix.values
raw_obs_matrix = raw_obs_matrix[:,:-1]

In [ ]:
model_2x = model_1_estimation(raw_obs_matrix[:366], QIj_vect, combis) # same model as model 1, but applied on raw data

### Model 4

In [ ]:
model_4x = []
for k in range(7):
    this_NT_hat_combis = model_4_estimation(obs_matrix, obs_matrix[:366], k, QIj_vect, combis)    
    model_4x.append(this_NT_hat_combis)

### Model 5

In [ ]:
model_5x = []
for k in range(7):
    this_NT_hat_combis = model_5_estimation(obs_matrix, obs_matrix[:366], k, QIj_vect, combis)
    model_5x.append(this_NT_hat_combis)

### Model 6

In [ ]:
model_6x = []
for k in range(7):
    this_NT_hat_combis = model_6_estimation(obs_matrix, obs_matrix[:366], k, QIj_vect, combis)    
    model_6x.append(this_NT_hat_combis)

In [ ]:
output_folder = f'../../outputs/files/models/{wwtp}/'

export_files('model_1', model_1x, raw_data, output_folder)
export_files('model_2', model_2x, raw_data, output_folder)
export_files('model_4', model_4x, raw_data, output_folder)
export_files('model_5', model_5x, raw_data, output_folder)
export_files('model_6', model_6x, raw_data, output_folder)

### Model 3

In [ ]:
sev_glob = glob.glob(f'../../outputs/files/models/{wwtp}/*.csv')
filtered_sev_glob = []
for file in sev_glob:
    if 'model_2' in file:
        filtered_sev_glob.append(file)

In [ ]:
for file in filtered_sev_glob:
    file_tail = file.split('model_2_')[1]
    file_fixed_param = file_tail.split('_')[0]
    file_combination = file_tail.split('_')[1].split('.csv')[0]

    sub_data = pd.read_csv(file, sep=';')
    sub_data.dateStart = pd.to_datetime(sub_data.dateStart)
    sub_data['obs'] = np.log(sub_data.Nt_hat)

    scou = model_3_estimation(sub_data)

    sub_data['muX'] = scou.muX
    sub_data['CIL'] = scou.CIL
    sub_data['CIU'] = scou.CIU

    sub_data['muX'] = np.exp(scou.muX)
    sub_data['CIL'] = np.exp(scou.CIL)
    sub_data['CIU'] = np.exp(scou.CIU)

    sub_data = sub_data.rename(columns={'Nt_hat':'obs', 'muX':'Nt_hat'})

    output_filepath = file.replace('model_2', 'model_3')
    sub_data.to_csv(output_filepath, index=False, sep=";")

# Models from the litterature

### Been et al., 2014

In [ ]:
# the first step consists in estimating an average NH4 coefficient from literature review
# instead of basing our calculations on the coefficient of Been et al., which was derived from swiss data
# we chose the 10.3 g day cap-1 value justified in the manuscript
# basically, its simply our model_2_1_NH4

In [ ]:
raw_obs_matrix = raw_data.loc[::, ['NH4', 'DCO', 'DBO', 'NTK', 'NGL', 'PT', 'MES', 'plantVolume']]
for k in range(7):
    raw_obs_matrix.iloc[:, k] *= raw_obs_matrix.iloc[:, -1]

raw_obs_matrix = raw_obs_matrix.values
raw_obs_matrix = raw_obs_matrix[:,:-1]

this_NT_hat = model_been_et_al(raw_obs_matrix, QIj_vect)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_been_original.csv', index=False, sep=";")

### van Nuijs et al., 2011

In [ ]:
# van nuijs et al.
# average over NGL, P, COD, BOD
# and from noisy measurements
# so we need to take the estimates for those four parameters, and then to perform an arithmetic mean

In [ ]:
those_NT_hats = []
for component in [1, 2, 4, 5]:
    this_NT_hat = model_vn_et_al(raw_obs_matrix, QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = those_NT_hats.mean(axis=0)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_vn_original.csv', index=False, sep=";")

### Zheng et al., 2019

In [ ]:
# Zheng et al. : same model as before but with a weighted average instead of a simple airthmetic mean
# P = 0.54 PNH4 + 0.33 * PCOD + 0.14 * PTP

In [ ]:
theta = np.array([0.54, 0.33, 0.14])
those_NT_hats = []
for component in [0, 1, 5]:
    this_NT_hat = model_vn_et_al(raw_obs_matrix[:366], QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = np.dot(theta, those_NT_hats)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_zheng_original.csv', index=False, sep=";")

# Models from the litterature combined with our smoothing algorithm

In [ ]:
# evaluating smoothing impact
# replicating the same models, but we now take our smoothing process into account

### Modified Been et al., 2014

In [ ]:
this_NT_hat = model_been_et_al(obs_matrix, QIj_vect)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_been_modified.csv', index=False, sep=";")

### Modified van Nuijs et al., 2011

In [ ]:
those_NT_hats = []
for component in [1, 2, 4, 5]:
    this_NT_hat = model_vn_et_al(obs_matrix, QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = those_NT_hats.mean(axis=0)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_vn_modified.csv', index=False, sep=";")

### Modified Zheng et al., 2019

In [ ]:
theta = np.array([0.54, 0.33, 0.14])
those_NT_hats = []
for component in [0, 1, 5]:
    this_NT_hat = model_vn_et_al(obs_matrix, QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = np.dot(theta, those_NT_hats)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_zheng_modified.csv', index=False, sep=";")

# MAV

In [ ]:
wwtp = 'MAV'
raw_data = get_raw_data(wwtp, filepath)
obs_matrix = get_obs_matrix(wwtp, gather_CIs=False)
print(raw_data.shape, obs_matrix.shape)

In [ ]:
# 'NH4', 'DCO', 'DBO', 'NTK', 'NGL', 'PT', 'MES'
QIj_vect = np.array([10.3, 120, 60, 12.6, 12.6, 1.2, 60])

In [ ]:
elements = np.arange(obs_matrix.shape[1]).tolist()

combis = []
for r in range(1, len(elements)+1):
    combis.extend(itertools.combinations(elements, r))

### Model 1

In [ ]:
model_1x = model_1_estimation(obs_matrix[:366], QIj_vect, combis)

### Model 2

In [ ]:
raw_obs_matrix = raw_data.loc[::, ['NH4', 'DCO', 'DBO', 'NTK', 'NGL', 'PT', 'MES', 'plantVolume']]
for k in range(7):
    raw_obs_matrix.iloc[:, k] *= raw_obs_matrix.iloc[:, -1]

raw_obs_matrix = raw_obs_matrix.values
raw_obs_matrix = raw_obs_matrix[:,:-1]

In [ ]:
model_2x = model_1_estimation(raw_obs_matrix[:366], QIj_vect, combis) # same model as model 1, but applied on raw data

### Model 4

In [ ]:
model_4x = []
for k in range(7):
    this_NT_hat_combis = model_4_estimation(obs_matrix, obs_matrix[:366], k, QIj_vect, combis)    
    model_4x.append(this_NT_hat_combis)

### Model 5

In [ ]:
model_5x = []
for k in range(7):
    this_NT_hat_combis = model_5_estimation(obs_matrix, obs_matrix[:366], k, QIj_vect, combis)
    model_5x.append(this_NT_hat_combis)

### Model 6

In [ ]:
model_6x = []
for k in range(7):
    this_NT_hat_combis = model_6_estimation(obs_matrix, obs_matrix[:366], k, QIj_vect, combis)    
    model_6x.append(this_NT_hat_combis)

In [ ]:
output_folder = f'../../outputs/files/models/{wwtp}/'

export_files('model_1', model_1x, raw_data, output_folder)
export_files('model_2', model_2x, raw_data, output_folder)
export_files('model_4', model_4x, raw_data, output_folder)
export_files('model_5', model_5x, raw_data, output_folder)
export_files('model_6', model_6x, raw_data, output_folder)

### Model 3

In [ ]:
mav_glob = glob.glob(f'../../outputs/files/models/{wwtp}/*.csv')
filtered_mav_glob = []
for file in mav_glob:
    if 'model_2' in file:
        filtered_mav_glob.append(file)

In [ ]:
for file in filtered_mav_glob:
    file_tail = file.split('model_2_')[1]
    file_fixed_param = file_tail.split('_')[0]
    file_combination = file_tail.split('_')[1].split('.csv')[0]

    sub_data = pd.read_csv(file, sep=';')
    sub_data.dateStart = pd.to_datetime(sub_data.dateStart)
    sub_data['obs'] = np.log(sub_data.Nt_hat)

    scou = model_3_estimation(sub_data)

    sub_data['muX'] = scou.muX
    sub_data['CIL'] = scou.CIL
    sub_data['CIU'] = scou.CIU

    sub_data['muX'] = np.exp(scou.muX)
    sub_data['CIL'] = np.exp(scou.CIL)
    sub_data['CIU'] = np.exp(scou.CIU)

    sub_data = sub_data.rename(columns={'Nt_hat':'obs', 'muX':'Nt_hat'})

    output_filepath = file.replace('model_2', 'model_3')
    sub_data.to_csv(output_filepath, index=False, sep=";")

# Models from the litterature

### Been et al., 2014

In [ ]:
raw_obs_matrix = raw_data.loc[::, ['NH4', 'DCO', 'DBO', 'NTK', 'NGL', 'PT', 'MES', 'plantVolume']]
for k in range(7):
    raw_obs_matrix.iloc[:, k] *= raw_obs_matrix.iloc[:, -1]

raw_obs_matrix = raw_obs_matrix.values
raw_obs_matrix = raw_obs_matrix[:,:-1]

this_NT_hat = model_been_et_al(raw_obs_matrix, QIj_vect)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_been_original.csv', index=False, sep=";")

### van Nuijs et al., 2011

In [ ]:
those_NT_hats = []
for component in [1, 2, 4, 5]:
    this_NT_hat = model_vn_et_al(raw_obs_matrix, QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = those_NT_hats.mean(axis=0)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_vn_original.csv', index=False, sep=";")

### Zheng et al., 2019

In [ ]:
theta = np.array([0.54, 0.33, 0.14])
those_NT_hats = []
for component in [0, 1, 5]:
    this_NT_hat = model_vn_et_al(raw_obs_matrix[:366], QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = np.dot(theta, those_NT_hats)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_zheng_original.csv', index=False, sep=";")

# Models from the litterature combined with our smoothing algorithm

### Modified Been et al., 2014

In [ ]:
this_NT_hat = model_been_et_al(obs_matrix, QIj_vect)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_been_modified.csv', index=False, sep=";")

### Modified van Nuijs et al., 2011

In [ ]:
those_NT_hats = []
for component in [1, 2, 4, 5]:
    this_NT_hat = model_vn_et_al(obs_matrix, QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = those_NT_hats.mean(axis=0)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_vn_modified.csv', index=False, sep=";")

### Modified Zheng et al., 2019

In [ ]:
theta = np.array([0.54, 0.33, 0.14])
those_NT_hats = []
for component in [0, 1, 5]:
    this_NT_hat = model_vn_et_al(obs_matrix, QIj_vect, component)
    those_NT_hats.append(this_NT_hat)

those_NT_hats = np.array(those_NT_hats)
this_NT_hat = np.dot(theta, those_NT_hats)
output_data = raw_data.copy()
output_data['Nt_hat'] = this_NT_hat
output_data.to_csv(f'../../outputs/files/models/{wwtp}/model_zheng_modified.csv', index=False, sep=";")

# The proposed model

### CLICHY

In [ ]:
wwtp = 'CLICHY'
raw_data = get_raw_data(wwtp, filepath, limit_2020=False)
obs_matrix, obs_matrix_CIL, obs_matrix_CIU = get_obs_matrix(wwtp, limit_2020=False)
raw_data = raw_data.loc[(raw_data.dateStart>='2021-01-01')&(raw_data.dateStart<='2024-12-25')]
raw_data.reset_index(inplace=True, drop=True)


k = 4
components = [0, 1, 2, 3, 4]

years = [2021, 2022, 2023, 2024]
NT_hat_list, NT_hat_CIL_list, NT_hat_CIU_list = [], [], []

for year in years:
    those_indexes = raw_data.loc[raw_data.year==year].index.tolist()

    this_NT_hat = best_model_estimation(obs_matrix[those_indexes], obs_matrix[those_indexes], k, components, QIj_vect)
    this_NT_hat_CIL = best_model_estimation(obs_matrix[those_indexes], obs_matrix_CIL[those_indexes], k, components, QIj_vect)
    this_NT_hat_CIU = best_model_estimation(obs_matrix[those_indexes], obs_matrix_CIU[those_indexes], k, components, QIj_vect)

    NT_hat_list.append(this_NT_hat)
    NT_hat_CIL_list.append(this_NT_hat_CIL)
    NT_hat_CIU_list.append(this_NT_hat_CIU)

concat_NT_hat = np.array([])

for el in NT_hat_list:
    el = np.array(el)
    concat_NT_hat = np.hstack((concat_NT_hat, el))

concat_NT_hat_CIL = np.array([])

for el in NT_hat_CIL_list:
    el = np.array(el)
    concat_NT_hat_CIL = np.hstack((concat_NT_hat_CIL, el))

concat_NT_hat_CIU = np.array([])

for el in NT_hat_CIU_list:
    el = np.array(el)
    concat_NT_hat_CIU = np.hstack((concat_NT_hat_CIU, el))

In [ ]:
raw_data['Nt_hat'] = concat_NT_hat
raw_data['Nt_hat_CIL'] = concat_NT_hat_CIL
raw_data['Nt_hat_CIU'] = concat_NT_hat_CIU
raw_data.to_csv(f'../../outputs/files/models/{wwtp}/best_model.csv', index=False, sep=";")

### MAV

In [ ]:
wwtp = 'MAV'
raw_data = get_raw_data(wwtp, filepath, limit_2020=True)
obs_matrix, obs_matrix_CIL, obs_matrix_CIU = get_obs_matrix(wwtp, limit_2020=True)

k = 4
components = [0, 1, 2, 3, 4]

this_NT_hat = best_model_estimation(obs_matrix, obs_matrix, k, components, QIj_vect)
this_NT_hat_CIL = best_model_estimation(obs_matrix, obs_matrix_CIL, k, components, QIj_vect)
this_NT_hat_CIU = best_model_estimation(obs_matrix, obs_matrix_CIU, k, components, QIj_vect)

In [ ]:
raw_data['Nt_hat'] = this_NT_hat
raw_data['Nt_hat_CIL'] = this_NT_hat_CIL
raw_data['Nt_hat_CIU'] = this_NT_hat_CIU
raw_data.to_csv(f'../../outputs/files/models/{wwtp}/best_model.csv', index=False, sep=";")

### SEV

In [ ]:
wwtp = 'SEV'
raw_data = get_raw_data(wwtp, filepath, limit_2020=True)
obs_matrix, obs_matrix_CIL, obs_matrix_CIU = get_obs_matrix(wwtp, limit_2020=True)

k = 4
components = [0, 1, 2, 3, 4]

this_NT_hat = best_model_estimation(obs_matrix, obs_matrix, k, components, QIj_vect)
this_NT_hat_CIL = best_model_estimation(obs_matrix, obs_matrix_CIL, k, components, QIj_vect)
this_NT_hat_CIU = best_model_estimation(obs_matrix, obs_matrix_CIU, k, components, QIj_vect)

In [ ]:
raw_data['Nt_hat'] = this_NT_hat
raw_data['Nt_hat_CIL'] = this_NT_hat_CIL
raw_data['Nt_hat_CIU'] = this_NT_hat_CIU
raw_data.to_csv(f'../../outputs/files/models/{wwtp}/best_model.csv', index=False, sep=";")